# Bước 3: Chỉ Số Mạng (Network Metrics)

**Mục tiêu:** Tính toán đầy đủ các chỉ số đặc trưng của ordinal network: Permutation Entropy, Network Entropy, Motif Frequency, Spectral measures, Centrality measures, và phân tích theo thời gian (time-varying).

**Input:** `data/preprocessed_log_returns.csv`, `data/transition_matrices.pkl`, `data/optimal_params.csv`  
**Output:** `data/network_metrics_static.csv`, `data/network_metrics_rolling.csv`

In [11]:
import subprocess, sys
for pkg in ["pandas", "numpy", "matplotlib", "seaborn", "networkx", "scipy", "scikit-learn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("✓ Packages ready.")

✓ Packages ready.


In [2]:
import warnings; warnings.filterwarnings("ignore")
import itertools, pickle
import sys
from pathlib import Path
from math import factorial, log

import warnings; warnings.filterwarnings("ignore")

# Force Agg backend before any pyplot import
import matplotlib
import matplotlib.backends
matplotlib.use('Agg')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy import stats
from scipy.linalg import eigvals

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 110})
DATA_DIR = Path("data")

# ── Load data ────────────────────────────────────────────────────────────────
log_ret = pd.read_csv(DATA_DIR / "preprocessed_log_returns.csv",
                      index_col=0, parse_dates=True)
COINS = list(log_ret.columns)

with open(DATA_DIR / "transition_matrices.pkl", "rb") as f:
    tm_data = pickle.load(f)

params = pd.read_csv(DATA_DIR / "optimal_params.csv")
BEST_D   = int(params["d"].iloc[0])
BEST_TAU = int(params["tau"].iloc[0])

print(f"Loaded: {len(COINS)} coins, d={BEST_D}, τ={BEST_TAU}")
print(f"Log-return shape: {log_ret.shape}")

Loaded: 8 coins, d=3, τ=1
Log-return shape: (2210, 8)


In [5]:
# ── Re-define helpers (self-contained notebook) ──────────────────────────────
def encode_ordinal_patterns(series, d, tau):
    n = len(series)
    n_patterns = n - (d - 1) * tau
    if n_patterns <= 0:
        return np.empty((0, d), dtype=np.int64)
    patterns = []
    for i in range(n_patterns):
        window = series[i: i + d * tau: tau]
        patterns.append(list(np.argsort(np.argsort(window)).astype(int)))
    return np.array(patterns, dtype=np.int64)


def permutation_entropy(patterns, d, normalize=True):
    if len(patterns) == 0:
        return np.nan
    arr = np.asarray(patterns, dtype=np.int64)
    unique, counts = np.unique(arr, axis=0, return_counts=True)
    probs = counts / counts.sum()
    H = -np.sum(probs * np.log(probs + 1e-12))
    return H / log(factorial(d)) if normalize else H


def network_entropy(T_matrix, normalize=True):
    """Shannon entropy of the stationary distribution of the Markov chain."""
    n = T_matrix.shape[0]
    # Stationary distribution via eigenvalue
    evals, evecs = np.linalg.eig(T_matrix.T)
    idx = np.argmin(np.abs(evals - 1.0))
    stationary = np.abs(evecs[:, idx])
    stationary /= stationary.sum() + 1e-12
    H = -np.sum(stationary * np.log(stationary + 1e-12))
    return H / log(n) if normalize else H


def motif_frequency(patterns, d):
    """Return dict of {pattern: frequency} for all d! patterns."""
    all_perms = list(itertools.permutations(range(d)))
    arr = np.asarray(patterns, dtype=np.int64)
    unique, counts = np.unique(arr, axis=0, return_counts=True)
    total = counts.sum()
    freq = {p: 0.0 for p in all_perms}
    for p, c in zip([tuple(u) for u in unique], counts):
        if p in freq:
            freq[p] = c / total
    return freq


print("Helper functions defined.")

Helper functions defined.


## 1 · Static Metrics (Full Sample)

In [6]:
def compute_spectral_metrics(T_matrix):
    evals = np.abs(eigvals(T_matrix))
    evals_sorted = np.sort(evals)[::-1]
    return {
        "spectral_gap": float(evals_sorted[0] - evals_sorted[1]) if len(evals_sorted) > 1 else 0.0,
        "spectral_radius": float(evals_sorted[0]),
        "second_eigenvalue": float(evals_sorted[1]) if len(evals_sorted) > 1 else 0.0,
        "mixing_rate": float(evals_sorted[1]) if len(evals_sorted) > 1 else 0.0,
    }


def compute_centrality_metrics(G: nx.DiGraph):
    if G.number_of_nodes() == 0:
        return {}
    pr   = nx.pagerank(G, weight="weight", alpha=0.85)
    deg  = dict(G.degree(weight="weight"))
    try:
        bet = nx.betweenness_centrality(G, weight="weight", normalized=True)
    except Exception:
        bet = {n: 0.0 for n in G.nodes()}
    return {
        "max_pagerank":    max(pr.values()),
        "mean_pagerank":   np.mean(list(pr.values())),
        "std_pagerank":    np.std(list(pr.values())),
        "max_degree":      max(deg.values()),
        "mean_degree":     np.mean(list(deg.values())),
        "max_betweenness": max(bet.values()),
        "mean_betweenness":np.mean(list(bet.values())),
        "density":         nx.density(G),
    }


static_metrics = []
for coin in COINS:
    series   = log_ret[coin].values
    patterns = encode_ordinal_patterns(series, BEST_D, BEST_TAU)
    T        = tm_data[coin]["T"]
    idx2pat  = tm_data[coin]["idx2pat"]

    # Build graph
    G = nx.DiGraph()
    n = T.shape[0]
    for i, pat in enumerate(idx2pat):
        G.add_node(i, label=str(pat))
    for i in range(n):
        for j in range(n):
            if T[i, j] >= 0.01:
                G.add_edge(i, j, weight=T[i, j])

    pe  = permutation_entropy(patterns, BEST_D)
    ne  = network_entropy(T)
    mf  = motif_frequency(patterns, BEST_D)
    sp  = compute_spectral_metrics(T)
    cen = compute_centrality_metrics(G)

    # Forbidden patterns (zero frequency patterns)
    n_forbidden = sum(1 for v in mf.values() if v == 0.0)

    row = {"coin": coin,
           "PE": round(pe, 5),
           "NE": round(ne, 5),
           "n_forbidden_patterns": n_forbidden,
           "n_patterns_total": factorial(BEST_D)}
    row.update({k: round(v, 5) for k, v in sp.items()})
    row.update({k: round(v, 5) for k, v in cen.items()})
    static_metrics.append(row)

static_df = pd.DataFrame(static_metrics)
print("=== Static Network Metrics ===")
display(static_df)

=== Static Network Metrics ===


,coin,PE,NE,n_forbidden_patterns,n_patterns_total,spectral_gap,spectral_radius,second_eigenvalue,mixing_rate,max_pagerank,mean_pagerank,std_pagerank,max_degree,mean_degree,max_betweenness,mean_betweenness,density
0,ADA,0.99946,0.99947,0,6,0.51918,1.0,0.48082,0.48082,0.17430,0.16667,0.00582,2.05620,2.0,0.20,0.12500,0.6
1,BNB,0.99981,0.99979,0,6,0.50544,1.0,0.49456,0.49456,0.16992,0.16667,0.00391,2.03408,2.0,0.20,0.14167,0.6
2,BTC,0.99977,0.99978,0,6,0.53406,1.0,0.46594,0.46594,0.17120,0.16667,0.00404,2.05164,2.0,0.20,0.14167,0.6
3,DOGE,0.99992,0.99992,0,6,0.51362,1.0,0.48638,0.48638,0.17031,0.16667,0.00241,2.03430,2.0,0.20,0.15000,0.6
4,ETH,0.99976,0.99977,0,6,0.49098,1.0,0.50902,0.50902,0.17098,0.16667,0.00401,2.03795,2.0,0.25,0.11667,0.6
5,LTC,0.99952,0.99952,0,6,0.49459,1.0,0.50541,0.50541,0.17275,0.16667,0.00565,2.05016,2.0,0.20,0.15000,0.6
6,SOL,0.99979,0.99979,0,6,0.52160,1.0,0.47840,0.47840,0.17091,0.16667,0.00392,2.04755,2.0,0.20,0.11667,0.6
7,XRP,0.99955,0.99955,0,6,0.52172,1.0,0.47828,0.47828,0.17631,0.16667,0.00565,2.06841,2.0,0.20,0.12500,0.6


## 2 · Motif Frequency Analysis

In [7]:
all_perms = list(itertools.permutations(range(BEST_D)))
motif_data = {}
for coin in COINS:
    series   = log_ret[coin].values
    patterns = encode_ordinal_patterns(series, BEST_D, BEST_TAU)
    mf       = motif_frequency(patterns, BEST_D)
    motif_data[coin] = [mf[p] for p in all_perms]

motif_df = pd.DataFrame(
    motif_data,
    index=[str(p) for p in all_perms]
).T   # (coins x patterns)

# Clustered heatmap
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(motif_df, annot=True, fmt=".3f", cmap="YlOrRd",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Frequency"})
ax.set_title(f"Ordinal Pattern (Motif) Frequencies  (d={BEST_D}, τ={BEST_TAU})",
             fontsize=12)
ax.set_xlabel("Pattern"); ax.set_ylabel("Coin")
ax.tick_params(axis="x", rotation=60, labelsize=8)
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_motif_frequencies.png", bbox_inches="tight")
plt.show()

# Uniform reference line
uniform = 1 / factorial(BEST_D)
print(f"Uniform frequency reference: {uniform:.4f}")
print("\nDeviation from uniform (abs):")
display((motif_df - uniform).abs().mean(axis=0).sort_values(ascending=False).to_frame("mean_abs_dev").T)

Uniform frequency reference: 0.1667

Deviation from uniform (abs):


,"(0, 1, 2)","(0, 2, 1)","(1, 2, 0)","(1, 0, 2)","(2, 1, 0)","(2, 0, 1)"
mean_abs_dev,0.009001,0.004076,0.003736,0.00368,0.00368,0.001755


## 3 · Chỉ số so sánh đa đồng tiền (Radar / Bar charts)

In [8]:
key_metrics = ["PE", "NE", "spectral_gap", "density", "mean_pagerank", "max_betweenness"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
colors = plt.cm.tab10(np.linspace(0, 1, len(COINS)))

for ax, metric in zip(axes.flat, key_metrics):
    vals = static_df.set_index("coin")[metric]
    bars = ax.bar(vals.index, vals.values, color=colors)
    ax.set_title(metric, fontsize=11)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
    for bar, v in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                f"{v:.3f}", ha="center", va="bottom", fontsize=8)

fig.suptitle("Network Metrics Comparison Across 8 Cryptocurrencies", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_metrics_comparison.png", bbox_inches="tight")
plt.show()

## 4 · Rolling (Time-varying) Network Metrics

In [9]:
ROLL_WIN = 180
STEP     = 30
all_perms = list(itertools.permutations(range(BEST_D)))
pat2idx   = {p: i for i, p in enumerate(all_perms)}
n_perms   = factorial(BEST_D)

rolling_rows = []
for coin in COINS:
    series = log_ret[coin].values
    dates  = log_ret.index

    for start in range(0, len(series) - ROLL_WIN - (BEST_D - 1) * BEST_TAU, STEP):
        window  = series[start: start + ROLL_WIN]
        pats    = encode_ordinal_patterns(window, BEST_D, BEST_TAU)
        if len(pats) < n_perms:
            continue

        # Build transition matrix for this window
        count_T = np.zeros((n_perms, n_perms))
        for k in range(len(pats) - 1):
            pi_now  = tuple(pats[k])
            pi_next = tuple(pats[k + 1])
            if pi_now in pat2idx and pi_next in pat2idx:
                count_T[pat2idx[pi_now], pat2idx[pi_next]] += 1
        row_sums = count_T.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        T_win = count_T / row_sums

        pe   = permutation_entropy(pats, BEST_D)
        ne   = network_entropy(T_win)
        sp   = compute_spectral_metrics(T_win)

        rolling_rows.append({
            "coin": coin,
            "date": dates[start + ROLL_WIN - 1],
            "PE":   pe,
            "NE":   ne,
            **{k: v for k, v in sp.items()},
        })

rolling_metrics_df = pd.DataFrame(rolling_rows)
rolling_metrics_df["date"] = pd.to_datetime(rolling_metrics_df["date"])
print(f"Rolling metrics shape: {rolling_metrics_df.shape}")

# Plot rolling PE vs NE for BTC
btc = rolling_metrics_df[rolling_metrics_df["coin"] == "BTC"]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax1.plot(btc["date"], btc["PE"], color="steelblue", lw=1.2, label="PE")
ax1.set_ylabel("Permutation Entropy"); ax1.legend()
ax2.plot(btc["date"], btc["NE"], color="darkorange", lw=1.2, label="Network Entropy")
ax2.set_ylabel("Network Entropy"); ax2.legend()
ax2.set_xlabel("Date")
fig.suptitle("BTC – Rolling Network Metrics", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_rolling_metrics_BTC.png", bbox_inches="tight")
plt.show()

Rolling metrics shape: (544, 8)


In [10]:
# Save outputs
static_df.to_csv(DATA_DIR / "network_metrics_static.csv", index=False)
print(f"✓ Saved network_metrics_static.csv  shape={static_df.shape}")

rolling_metrics_df.to_csv(DATA_DIR / "network_metrics_rolling.csv", index=False)
print(f"✓ Saved network_metrics_rolling.csv  shape={rolling_metrics_df.shape}")

motif_df.to_csv(DATA_DIR / "motif_frequencies.csv")
print(f"✓ Saved motif_frequencies.csv  shape={motif_df.shape}")

print("\n=== Network Metrics complete ===")

✓ Saved network_metrics_static.csv  shape=(8, 17)
✓ Saved network_metrics_rolling.csv  shape=(544, 8)
✓ Saved motif_frequencies.csv  shape=(8, 6)

=== Network Metrics complete ===
